# MVP pipeline imágenes de repuestos (DONREP)

## Etapa 0 — Carga de datos

Librerias 

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path


In [2]:
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

INPUT_XLSX = ROOT / "input" / "products.xlsx"
COL_REF, COL_BRAND, COL_NAME = "Ref Proveedor", "Marca", "Nombre"

df = pd.read_excel(INPUT_XLSX, dtype={COL_REF: str})
df = df[[COL_REF, COL_BRAND, COL_NAME]].copy()
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()
df = df[df[COL_REF].notna() & (df[COL_REF] != "") & (df[COL_REF] != "nan")].reset_index(drop=True)

print(f"{len(df)} productos cargados")
df.head(10)

281 productos cargados


,Ref Proveedor,Marca,Nombre
0,8550504748,MOTRIO,10W30 MOTRIO 1LX12UN
1,8550504747,MOTRIO,10W30 MOTRIO 4LX4UND
2,7711737740,MOTRIO,10W30 MOTRIO 4X4 UNID
3,8550504765,MOTRIO,10W40 MOTRIO A3/B4 SS 1LX12UND
4,8550504764,MOTRIO,10W40 MOTRIO A3/B4 SS 4LX4UND
5,8550504745,MOTRIO,15W40 MOTRIO 1LX12UN
6,7711737737,MOTRIO,15W40 MOTRIO 4LX4UND
7,8550504744,MOTRIO,15W40 MOTRIO 4LX4UND
8,7711649209,RENAULT,185/65R15 ENERGY XM2
9,8550504751,MOTRIO,20W50 MOTRIO 1LX12UN


## Etapa 1 — Normalización de nombres

Es importante revisar visualmente `nombre_original` vs `nombre_limpio` / `termino_busqueda` luego de la normalizacion. De ser necesario se puede ajustar `config/abbreviations.py` .

In [3]:
from pipeline.normalize import normalize_df

df_norm = normalize_df(df, col_nombre=COL_NAME, col_marca=COL_BRAND)

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 300)
df_norm[["nombre_original", "nombre_limpio", "termino_busqueda"]].head(10)

,nombre_original,nombre_limpio,termino_busqueda
0,10W30 MOTRIO 1LX12UN,10W30 MOTRIO 1L,10W30 MOTRIO 1L
1,10W30 MOTRIO 4LX4UND,10W30 MOTRIO 4L,10W30 MOTRIO 4L
2,10W30 MOTRIO 4X4 UNID,10W30 MOTRIO 4X4 UND,10W30 MOTRIO 4X4 UND
3,10W40 MOTRIO A3/B4 SS 1LX12UND,10W40 MOTRIO A3/B4 SS 1L,10W40 MOTRIO A3/B4 SS 1L
4,10W40 MOTRIO A3/B4 SS 4LX4UND,10W40 MOTRIO A3/B4 SS 4L,10W40 MOTRIO A3/B4 SS 4L
5,15W40 MOTRIO 1LX12UN,15W40 MOTRIO 1L,15W40 MOTRIO 1L
6,15W40 MOTRIO 4LX4UND,15W40 MOTRIO 4L,15W40 MOTRIO 4L
7,15W40 MOTRIO 4LX4UND,15W40 MOTRIO 4L,15W40 MOTRIO 4L
8,185/65R15 ENERGY XM2,185/65R15 ENERGY XM2,185/65R15 ENERGY XM2 RENAULT
9,20W50 MOTRIO 1LX12UN,20W50 MOTRIO 1L,20W50 MOTRIO 1L


In [4]:
# columnas estructuradas extraídas
df_norm[["nombre_original", "pack", "litraje", "pack_ambiguo", "viscosidad", "medida_llanta", "vehiculo_code_raw"]].head(10)

,nombre_original,pack,litraje,pack_ambiguo,viscosidad,medida_llanta,vehiculo_code_raw
0,10W30 MOTRIO 1LX12UN,1LX12UN,1L,False,10W30,,
1,10W30 MOTRIO 4LX4UND,4LX4UND,4L,False,10W30,,
2,10W30 MOTRIO 4X4 UNID,4X4 UND,,True,10W30,,UND
3,10W40 MOTRIO A3/B4 SS 1LX12UND,1LX12UND,1L,False,10W40,,
4,10W40 MOTRIO A3/B4 SS 4LX4UND,4LX4UND,4L,False,10W40,,
5,15W40 MOTRIO 1LX12UN,1LX12UN,1L,False,15W40,,
6,15W40 MOTRIO 4LX4UND,4LX4UND,4L,False,15W40,,
7,15W40 MOTRIO 4LX4UND,4LX4UND,4L,False,15W40,,
8,185/65R15 ENERGY XM2,,,False,,185/65R15,XM2
9,20W50 MOTRIO 1LX12UN,1LX12UN,1L,False,20W50,,


In [5]:
df_norm[["nombre_original", "pack", "litraje", "pack_ambiguo", "viscosidad", "medida_llanta", "vehiculo_code_raw"]].tail(10)

,nombre_original,pack,litraje,pack_ambiguo,viscosidad,medida_llanta,vehiculo_code_raw
271,VIDRIO LUNETA TR SA3,,,False,,,SA3
272,VIDRIO MOVIL PUERTA NDU,,,False,,,NDU
273,VIDRIO MOVIL PUERTA TRASERA DERECHA X52,,,False,,,X52
274,VIDRIO MOVIL PUERTA TRASERA IZQUIER GKO,,,False,,,GKO
275,VIDRIO PARABRISA NDU,,,False,,,NDU
276,VIDRIO PARABRISAS NKG,,,False,,,NKG
277,VIDRIO PARABRISAS OR2,,,False,,,OR2
278,VIDRIO PARABRISAS SIN SENSOR LLUVIA NDU,,,False,,,NDU
279,VIDRIO PTA TR IZ SA3,,,False,,,SA3
280,VIDRIO PUERTA DELANTERA DERECHA NDUH,,,False,,,NDUH


### Filas de pack ambiguo (revisar a mano)

Tienen expresión de pack pero sin "L" explícito, así que no se pudo separar litraje de multiplicador de caja - quedaron intactas en `nombre_limpio`. ¿`4X4 UNID` es un típo de `4LX4UND`? ¿Qué significan `3X5` / `6X1` de Castrol?

In [6]:
df_norm[df_norm["pack_ambiguo"]][["nombre_original", "nombre_limpio", "pack"]]

,nombre_original,nombre_limpio,pack
2,10W30 MOTRIO 4X4 UNID,10W30 MOTRIO 4X4 UND,4X4 UND
24,AMOR TRA MOT 4X4 DU,AMORTIGUADOR TRASERO MOTOR 4X4 DU,4X4
65,CASTROL 10W40 3X5,CASTROL 10W40 3X5,3X5
66,CASTROL 10W40 6X1,CASTROL 10W40 6X1,6X1
67,CASTROL 10W40 6X1,CASTROL 10W40 6X1,6X1
68,CASTROL 15W40 3X5,CASTROL 15W40 3X5,3X5
69,CASTROL 15W40 3X5,CASTROL 15W40 3X5,3X5
70,CASTROL 15W40 6X1,CASTROL 15W40 6X1,6X1
71,CASTROL 15W40 6X1,CASTROL 15W40 6X1,6X1
72,CASTROL 20W50 3X5,CASTROL 20W50 3X5,3X5


## Etapa 2 — Categorización de productos, con base en los nombres


In [7]:
from pipeline.categorize import categorize_df, coverage_report
coverage_report(df_norm)          # prints per-category counts + the `otros` backlog list
cd = categorize_df(df_norm)       # adds categoria / cse_profile / presentacion columns
cd[["nombre_limpio", "categoria", "cse_profile"]].head(10)

Cobertura: 281 filas, 15 categorias

   57  carroceria
   38  vidrios_espejos
   33  iluminacion
   29  lubricantes
   21  suspension_direccion
   20  merchandising
   17  frenos
   13  motor_transmision
   13  emblemas
   10  filtros
    9  refrigeracion
    7  llantas_rines
    7  baterias
    4  otros
    3  accesorios

'otros' (backlog para nuevas reglas): 4 (1%)
    - KIT JUNTA DECANTADOR NT2
    - KIT JUNTAS INDICADOR CARBURANTE
    - MODULO VARIA CLIM NM
    - PLASTIGLO 250ML TT


,nombre_limpio,categoria,cse_profile
0,10W30 MOTRIO 1L,lubricantes,baseline
1,10W30 MOTRIO 4L,lubricantes,baseline
2,10W30 MOTRIO 4X4 UND,lubricantes,baseline
3,10W40 MOTRIO A3/B4 SS 1L,lubricantes,baseline
4,10W40 MOTRIO A3/B4 SS 4L,lubricantes,baseline
5,15W40 MOTRIO 1L,lubricantes,baseline
6,15W40 MOTRIO 4L,lubricantes,baseline
7,15W40 MOTRIO 4L,lubricantes,baseline
8,185/65R15 ENERGY XM2,llantas_rines,baseline
9,20W50 MOTRIO 1L,lubricantes,baseline


## Etapa 3 — Búsqueda CSE

Una llamada por producto. **Rung 1 (`"{ref}" "{marca}"`, ambas comilladas)** es el default: la query mínima y probada. La respuesta cruda del CSE se cachea en `cache/cse/{ref}/rung1.json`, así que iterar el resto del pipeline **no vuelve a pagar** búsquedas.

Perfil por categoría → params de imagen: `baseline` = `imgType=photo,imgSize=large`; `white_dominant` añade `imgDominantColor=white`; `exact_brand` añade `exactTerms={marca}`. Sesgo de mercado `gl=co&hl=es`.

Los rungs 2–5 (sin comillas / `nombre_limpio` / etc.) existen en `pipeline/search.py` pero **no** se auto-escalan todavía

In [8]:
# Golden set: ~1-2 productos por categoria para probar la busqueda sin gastar
# en las 281 filas. Se amplia/ajusta a mano segun lo que se quiera revisar.
golden = cd.groupby("categoria", group_keys=False).head(2).reset_index(drop=True)

print(f"Golden set: {len(golden)} productos, {golden['categoria'].nunique()} categorias")
golden[["Ref Proveedor", "Marca", "nombre_limpio", "categoria", "cse_profile"]]


Golden set: 30 productos, 15 categorias


,Ref Proveedor,Marca,nombre_limpio,categoria,cse_profile
0,8550504748,MOTRIO,10W30 MOTRIO 1L,lubricantes,baseline
1,8550504747,MOTRIO,10W30 MOTRIO 4L,lubricantes,baseline
2,7711649209,RENAULT,185/65R15 ENERGY XM2,llantas_rines,baseline
3,631019973R,RENAULT,ALETA DELANTERO IZQUIERDO SA3-LN3,carroceria,baseline
4,631000689R,RENAULT,ALETA DELANTERO DERECHO CP,carroceria,baseline
5,8660005993,RENAULT,AMORTIGUADOR TRASERO MOTOR 4X4 DU,suspension_direccion,baseline
6,543022720R,RENAULT,AMORTIGUADOR DELANTERO OR,suspension_direccion,baseline
7,D40604JA0A,RENAULT,BANDAS FRENO TRASERO ALASKAN,frenos,baseline
8,03-T4770,RENAULT,BATERIA,baterias,baseline
9,244103090R,RENAULT,BATERIA 14AH TWIZY,baterias,baseline


In [9]:
# rung = cual query construir:  1 -> "ref" "marca" (default) | 2 -> ref marca | 3 -> nombre_limpio
# max_queries = tope de llamadas NUEVAS a la API (protege las 100 gratis/dia).
# La respuesta cruda queda en cache/cse/{ref}/rung{n}.json -> re-correr no re-paga.
from pipeline.search import search_df

resultados = search_df(golden, rung=1, max_queries=40)

# Resumen: cuantos candidatos trajo cada producto y de que dominios
resumen = pd.DataFrame([
    {"ref": ref,
     "n_candidatos": len(cands),
     "dominios": ", ".join(sorted({c["displayLink"] for c in cands})[:5])}
    for ref, cands in resultados.items()
])
resumen


rung 1: 30 queries nuevas a la API (cap 40), 0 desde cache, 0 omitidas por el cap


,ref,n_candidatos,dominios
0,8550504748,7,"motrio.com.co, www.imotriz.com"
1,8550504747,6,"motrio.com.co, www.imotriz.com"
2,7711649209,10,"listado.mercadolibre.com.co, listado.tucarro.com.co"
3,631019973R,10,"avtopro.es, partsouq.com, www.dts.lv, www.pieseautoarges..."
4,631000689R,10,"boodmo.com, loja.rpoint.com.br, podkapotom.ru, www.b-par..."
5,8660005993,10,"avto.pro, avtopro.es, www.autodoc.parts, www.autodoc24.f..."
6,543022720R,10,"loja.grupoamazonas.com.br, www.jotacarautopecas.com.br, ..."
7,D40604JA0A,10,"club.autodoc.co.uk, www.autodoc.co.uk, www.ebay.co.uk, w..."
8,03-T4770,0,
9,244103090R,10,"baterias.com, www.autodoc.parts, www.b-parts.com, www.de..."


In [10]:
# Preview inline: miniaturas por producto para revisar a ojo el pool crudo
from IPython.display import HTML, display

for ref, cands in resultados.items():
    fila = golden[golden["Ref Proveedor"] == ref].iloc[0]
    print(f"{ref}  |  {fila['nombre_limpio']}  |  {fila['categoria']} "
        f"({fila['cse_profile']})  |  {len(cands)} candidatos")
    imgs = "".join(
        f'<img src="{c["thumbnailLink"]}" title="{c["displayLink"]}" '
        f'style="height:90px;margin:2px;border:1px solid #ccc">'
        for c in cands if c["thumbnailLink"]
    )
    display(HTML(imgs or "<i>(sin candidatos)</i>"))


8550504748  |  10W30 MOTRIO 1L  |  lubricantes (baseline)  |  7 candidatos


8550504747  |  10W30 MOTRIO 4L  |  lubricantes (baseline)  |  6 candidatos


7711649209  |  185/65R15 ENERGY XM2  |  llantas_rines (baseline)  |  10 candidatos


631019973R  |  ALETA DELANTERO IZQUIERDO SA3-LN3  |  carroceria (baseline)  |  10 candidatos


631000689R  |  ALETA DELANTERO DERECHO CP  |  carroceria (baseline)  |  10 candidatos


...

8671002274  |  FILTRO ACEITE MOTRI TT  |  filtros (baseline)  |  10 candidatos


118301718R  |  KIT JUNTA DECANTADOR NT2  |  otros (baseline)  |  10 candidatos


7701207449  |  KIT JUNTAS INDICADOR CARBURANTE  |  otros (baseline)  |  10 candidatos


749M65735R  |  TAPETE TERMOFORMADO MILDH NDUH  |  accesorios (baseline)  |  10 candidatos


749M65098R  |  TAPETE TERMOFORMADO NDUH  |  accesorios (baseline)  |  10 candidatos


## Etapa 4 — Pre-filtro (sin Gemini)

Sobre el pool crudo de la Etapa 3: dedupe por URL y por contenido visual (perceptual hash —> misma foto servida desde dos dominios distintos), descarta resolución baja / aspect ratio de banner / dominios en `config/domains.py`, y luego **ordena** (no descarta) por dominio prioritario + score de fondo blanco calculado del thumbnail. Los primeros `keep=6` que sobreviven se verifican con un HEAD (sin bajar la imagen completa —> Gemini la pedirá después, en memoria).

Todo el proceso escribe **solo un JSON de resultado por producto** en `cache/prefilter/{ref}.json` — no se guardan imágenes a disco, así que re-correr esta celda no vuelve a pegarle a la red.

In [11]:
# Corre el pre-filtro sobre el pool de la Etapa 3 (golden set). use_cache=True
# por defecto en prefilter_product: re-correr esta celda no re-hace los HEAD
# ni vuelve a bajar thumbnails, sirve el JSON de cache/prefilter/{ref}.json.
from pipeline.prefilter import prefilter_product

filtrados = {ref: prefilter_product(ref, cands) for ref, cands in resultados.items()}

resumen_prefiltro = pd.DataFrame([
    {"ref": ref,
     "n_input": r["n_input"],
     "n_kept": len(r["kept"]),
     "n_dropped": len(r["dropped"])}
    for ref, r in filtrados.items()
])
resumen_prefiltro


,ref,n_input,n_kept,n_dropped
0,8550504748,7,2,5
1,8550504747,6,2,4
2,7711649209,10,0,10
3,631019973R,10,6,1
4,631000689R,10,4,6
5,8660005993,10,6,4
6,543022720R,10,6,3
7,D40604JA0A,10,6,4
8,03-T4770,0,0,0
9,244103090R,10,2,8


In [12]:
# Preview inline: candidatos sobrevivientes (con su bg_score) y por que se
# cayeron los demas - clave para calibrar MIN_SIDE / MAX_ASPECT_RATIO /
# HASH_MAX_DISTANCE y para ir llenando config/domains.py con evidencia real.
for ref, r in filtrados.items():
    fila = golden[golden["Ref Proveedor"] == ref].iloc[0]
    print(f"{ref}  |  {fila['nombre_limpio']}  |  {fila['categoria']}  "
        f"|  {len(r['kept'])}/{r['n_input']} sobrevivieron")

    imgs = "".join(
        f'<div style="display:inline-block;text-align:center;margin:2px">'
        f'<img src="{c["thumbnailLink"]}" title="{c["displayLink"]}" '
        f'style="height:90px;border:1px solid #ccc"><br>'
        f'<small>{c["bg_score"]}</small></div>'
        for c in r["kept"] if c["thumbnailLink"]
    )
    display(HTML(imgs or "<i>(sin candidatos sobrevivientes)</i>"))

    if r["dropped"]:
        razones = pd.Series([reason for _, reason in r["dropped"]]).value_counts()
        print("  descartados:", dict(razones))


8550504748  |  10W30 MOTRIO 1L  |  lubricantes  |  2/7 sobrevivieron


  descartados: {'resolucion_baja 300x205': np.int64(5)}
8550504747  |  10W30 MOTRIO 4L  |  lubricantes  |  2/6 sobrevivieron


  descartados: {'resolucion_baja 300x205': np.int64(4)}
7711649209  |  185/65R15 ENERGY XM2  |  llantas_rines  |  0/10 sobrevivieron


  descartados: {'resolucion_baja 320x320': np.int64(10)}
631019973R  |  ALETA DELANTERO IZQUIERDO SA3-LN3  |  carroceria  |  6/10 sobrevivieron


  descartados: {'resolucion_baja 340x227': np.int64(1)}
631000689R  |  ALETA DELANTERO DERECHO CP  |  carroceria  |  4/10 sobrevivieron


...

  descartados: {'resolucion_baja 220x220': np.int64(1), 'resolucion_baja 379x400': np.int64(1), 'resolucion_baja 265x177': np.int64(1), 'resolucion_baja 357x400': np.int64(1)}
118301718R  |  KIT JUNTA DECANTADOR NT2  |  otros  |  4/10 sobrevivieron


  descartados: {'resolucion_baja 390x295': np.int64(1), 'duplicado_visual': np.int64(1)}
7701207449  |  KIT JUNTAS INDICADOR CARBURANTE  |  otros  |  4/10 sobrevivieron


  descartados: {'duplicado_visual': np.int64(2), 'resolucion_baja 480x357': np.int64(1), 'resolucion_baja 350x291': np.int64(1)}
749M65735R  |  TAPETE TERMOFORMADO MILDH NDUH  |  accesorios  |  5/10 sobrevivieron


  descartados: {'resolucion_baja 480x360': np.int64(2), 'resolucion_baja 528x396': np.int64(1), 'resolucion_baja 400x264': np.int64(1), 'resolucion_baja 228x228': np.int64(1)}
749M65098R  |  TAPETE TERMOFORMADO NDUH  |  accesorios  |  4/10 sobrevivieron


  descartados: {'resolucion_baja 228x228': np.int64(1), 'resolucion_baja 400x264': np.int64(1), 'resolucion_baja 500x347': np.int64(1), 'resolucion_baja 500x351': np.int64(1), 'resolucion_baja 375x211': np.int64(1)}
